In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import pytorch_lightning as pl


from domains.previsao_ceu.refactored_module import build_preprocessing_pipeline

In [3]:
import yaml

def load_config():
    with open('configs/ceu_config.yaml', 'r', encoding='utf-8') as f:
        return yaml.safe_load(f)

CONFIG_CEU = load_config()

In [5]:
# 2. Leitura
csv_path = CONFIG_CEU['csv_path']
print(f"⏳ Lendo: {csv_path}")
df = pd.read_csv(csv_path)

if 'Date_Time' in df.columns:
    df['Date_Time'] = pd.to_datetime(df['Date_Time'])
    df = df.drop_duplicates(subset=['Date_Time'], keep='first').set_index('Date_Time').sort_index()
df = df[~df.index.duplicated(keep='first')]

df['Temperatura ambiente °C'].loc[df['Temperatura ambiente °C'] < 0] = np.nan
df['Umidade Relativa %'].loc[df['Umidade Relativa %'] < 0] = np.nan

# 3. Pré-processamento
pp_conf = CONFIG_CEU['preprocessing']

# Instancia passando o mapa explícito. 
# Isso garante que a padronização aconteça conforme o CONFIG_CEU acima.
preprocessor = build_preprocessing_pipeline(
    latitude=pp_conf['latitude'], 
    longitude=pp_conf['longitude'], 
    altitude=pp_conf['altitude'],
    timezone=pp_conf['timezone'], 
    nominal_power=pp_conf['nominal_power'], 
    start_year=pp_conf['start_year'],
    features_to_scale=pp_conf['features_to_scale'],
    target_col=CONFIG_CEU['prediction_mode'], # <--- unica variavel que não vem do preprocessing
    column_mapping=pp_conf['column_mapping'],
    cs_model = 'esra',
    kasten_corr=True
)

preprocessor.fit(df)

# O método transform usa o column_mapping para renomear as colunas
df_processed = preprocessor.transform(df)

⏳ Lendo: data/pv0.csv
Otimizando TL para 381 dias selecionados...
Otimização concluída. Média TL: 6.28


In [7]:
df_cloud = pd.DataFrame()
b0 = 14.884
b1 = 10.184
"""
df_cloud['kt'] = np.arange(0,df_processed['kt'].max(),0.000001)
df_cloud['threshold'] = 1 / (1 + np.exp(b0*df_cloud['kt'] - b1))
"""

df_processed['cloud_enh'] = np.where(df_processed['fracao_difusa'] > (1 / (1 + np.exp(b0*df_processed['kt'] - b1))), 1, 0)
df_processed['QS2'] = 1 - np.sqrt((1-df_processed['kt'])**2 + df_processed['fracao_difusa']**2)/np.sqrt(2)
df_processed['QS3'] = 1 - np.sqrt((1-df_processed['kt'])**2 + df_processed['fracao_difusa']**2 + df_processed['VS_cdfn']**2)/np.sqrt(3)

df_processed['delta_theta']= np.arctan2(df_processed['fracao_difusa'].diff(),df_processed['kt'].diff())/(2*np.pi)
df_processed['raio'] = np.sqrt(df_processed['kt'].diff()**2 + df_processed['fracao_difusa'].diff()**2)

In [16]:
import pandas as pd
import numpy as np
import plotly.express as px
import umap.umap_ as umap

# Remover valores nulos (NaN) provenientes de lags no pré-processamento,
# pois o UMAP e o Plotly PCP exigem tabelas completas.
df_clean = df_processed.loc[(df_processed['mask'] == 1)].dropna().copy()
#df_clean = df_processed.dropna().copy()
# A sua coluna que reflete a Potência/Geração se chama "target" no df_processed
target_col = 'QS2'

# Definir as variáveis meteorológicas numéricas que desejamos analisar nativamente
meteo_cols_base = [
     'temp_amb', 'humidity', 'wind_speed', 'rain'
     ]
# Interseção com as colunas reais para evitar erros
meteo_cols = [col for col in meteo_cols_base if col in df_clean.columns]

# Transformar o index (Date_Time) em coluna regular para o "hover_data" e 
# separar o dataframe das variáveis apenas meteorológicas para o algortimo UMAP
df_vis = df_clean.reset_index()
X_meteo = df_clean[meteo_cols]


In [17]:
# PCP: Unindo as variáveis meteorológicas com o Target para servir de escala de cores
pcp_cols = meteo_cols + [target_col]

fig_pcp = px.parallel_coordinates(
    df_vis,
    dimensions=pcp_cols,
    color=target_col,
    color_continuous_scale=px.colors.sequential.Viridis,
    title='Gráfico de Coordenadas Paralelas (PCP) - Variáveis Meteorológicas vs Potência'
)

fig_pcp.show()


In [ ]:
# Treinando EXCLUSIVAMENTE nas variáveis meteorológicas escaladas
reducer = umap.UMAP(
    n_neighbors=50, 
    min_dist=0.2, 
    n_components=2, 
    random_state=42 # Fixar pseudoaleatoriedade para reprodutibilidade no XAI
)

# Ajuste e transformação
umap_embedding = reducer.fit_transform(X_meteo)

# Atribuir as componentes geradas ao dataframe de visualização
df_vis['UMAP_1'] = umap_embedding[:, 0]
df_vis['UMAP_2'] = umap_embedding[:, 1]
#df_vis['UMAP_3'] = umap_embedding[:, 2]


In [19]:
# Gráfico de Dispersão utilizando as componentes latentes criadas pelo UMAP
fig_umap = px.scatter(
    df_vis,
    x='UMAP_1',
    y='UMAP_2',
    color=target_col, # Cor representando a Potência/Geração do sistema
    hover_data=['Date_Time', target_col], # Hover rico para inspecionar agrupamentos/anomalias
    color_continuous_scale=px.colors.sequential.Plasma,
    title='Projeção UMAP 2D - Regimes Climáticos Latentes',
    labels={
        'UMAP_1': 'Componente 1 (UMAP)',
        'UMAP_2': 'Componente 2 (UMAP)',
        target_col: 'Kt'
    }
)

# Configurando o tamanho dos pontos para evitar "overplotting" de dados densos
fig_umap.update_traces(marker=dict(size=4, opacity=0.8))
fig_umap.show()


In [57]:

fig_umap = px.scatter_3d(
    df_vis,
    x='UMAP_1',
    y='UMAP_2',
    z='UMAP_3',
    color=target_col, 
    hover_data=['Date_Time', target_col],
    color_continuous_scale=px.colors.sequential.Plasma,
    title='Projeção UMAP 3D - Regimes Climáticos Latentes',
    labels={
        'UMAP_1': 'Componente 1',
        'UMAP_2': 'Componente 2',
        'UMAP_3': 'Componente 3',
        target_col: 'Fração Difusa (K)'
    }
)

# 2. Ajuste de marcadores (reduzi um pouco o tamanho para o 3D não poluir)
fig_umap.update_traces(marker=dict(size=3, opacity=0.7))

# 3. Ajuste de layout para melhorar a visualização 3D
fig_umap.update_layout(margin=dict(l=0, r=0, b=0, t=40))

fig_umap.show()


In [20]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

# A unidade analítica do MMD-Critic para o nosso problema climático deve ser o "Dia"
df_vis['Date'] = df_vis['Date_Time'].dt.date

# Extrair características estatísticas de cada dia para alimentar o Kernel RBF
daily_features = df_vis.groupby('Date').agg({
    'ghi': ['mean', 'max', 'std'],
    'temp_amb': ['mean', 'max', 'min'],
    'irrad_poa': ['mean', 'max'],
    'humidity': ['mean', 'std'],
    'wind_speed': ['mean', 'max']
})

# Achatar o multi-index das colunas (ex: ('ghi', 'max') vira 'ghi_max')
daily_features.columns = ['_'.join(col) for col in daily_features.columns]

# Preencher possíveis NaNs (ex: std de um dia que possui só 1 hora registrada) com 0
daily_features = daily_features.fillna(0)

# O Kernel RBF é sensível à escala. Como as agregações misturam grandezas (mean vs std), escalamos novamente
scaler_daily = StandardScaler()
X_daily = scaler_daily.fit_transform(daily_features)
dates = daily_features.index.values


In [21]:
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.cluster import KMeans
from sklearn.ensemble import IsolationForest
from sklearn.metrics import pairwise_distances_argmin_min

# ---- ABORDAGEM 1: MMD-Critic (Been Kim et al. 2016) ----
def get_mmd_prototypes(X, num_prototypes, gamma=None):
    """Seleção Gulosa (Greedy) de Protótipos minimizando o erro MMD."""
    n = len(X)
    K = rbf_kernel(X, gamma=gamma)
    col_sum = np.sum(K, axis=0) / n 
    
    selected = []
    candidates = list(range(n))
    
    for _ in range(num_prototypes):
        best_obj = float('inf')
        best_idx = -1
        for i in candidates:
            temp_selected = selected + [i]
            m = len(temp_selected)
            K_sub = K[np.ix_(temp_selected, temp_selected)]
            
            term1 = np.sum(K_sub) / (m * m)
            term2 = 2 * np.sum(col_sum[temp_selected]) / m
            obj = term1 - term2
            
            if obj < best_obj:
                best_obj = obj
                best_idx = i
        selected.append(best_idx)
        candidates.remove(best_idx)
    return selected

def get_mmd_criticisms(X, selected_prototypes, num_criticisms, gamma=None):
    """Extração de críticas selecionando pontos que maximizam a função witness (Anomalia MMD)."""
    n = len(X)
    m = len(selected_prototypes)
    K = rbf_kernel(X, gamma=gamma)
    
    col_sum = np.sum(K, axis=0) / n
    proto_sum = np.sum(K[:, selected_prototypes], axis=1) / m
    
    witness = np.abs(col_sum - proto_sum)
    witness[selected_prototypes] = -1 # Evita selecionar um protótipo como crítica
    
    selected_crit = np.argsort(witness)[-num_criticisms:][::-1]
    return selected_crit.tolist()


# ---- ABORDAGEM 2: K-Means e Isolation Forest (Alternativa Robusta / Fallback) ----
def get_alternative_summaries(X, dates, n=5):
    # Protótipos via KMeans (Aproximação eficiente de K-Medoids)
    kmeans = KMeans(n_clusters=n, random_state=42, n_init='auto').fit(X)
    proto_idx, _ = pairwise_distances_argmin_min(kmeans.cluster_centers_, X)
    
    # Críticas via Isolation Forest
    iso = IsolationForest(contamination=0.1, random_state=42).fit(X)
    scores = iso.decision_function(X) # Quanto menor, mais anômalo
    crit_idx = np.argsort(scores)[:n]
    
    return proto_idx, crit_idx


In [22]:
num_samples = 5

# O Parâmetro Gamma ótimo (heurística de dispersão do Kernel baseada no total de features)
gamma_val = 1.0 / X_daily.shape[1]

# Extração MMD Clássica
proto_idx_mmd = get_mmd_prototypes(X_daily, num_prototypes=num_samples, gamma=gamma_val)
crit_idx_mmd = get_mmd_criticisms(X_daily, proto_idx_mmd, num_criticisms=num_samples, gamma=gamma_val)

proto_dates = dates[proto_idx_mmd]
crit_dates = dates[crit_idx_mmd]

# Extração via Alternativa para debug e comparação de métodos
alt_proto_idx, alt_crit_idx = get_alternative_summaries(X_daily, dates, n=num_samples)

print("=== Resultados: MMD-Critic ===")
print("Dias Protótipos (Típicos):", proto_dates)
print("Dias Críticas (Anômalos): ", crit_dates)


=== Resultados: MMD-Critic ===
Dias Protótipos (Típicos): [datetime.date(2018, 4, 9) datetime.date(2021, 8, 16)
 datetime.date(2017, 11, 3) datetime.date(2018, 8, 18)
 datetime.date(2021, 9, 9)]
Dias Críticas (Anômalos):  [datetime.date(2022, 5, 9) datetime.date(2017, 2, 26)
 datetime.date(2018, 8, 9) datetime.date(2019, 4, 16)
 datetime.date(2016, 10, 5)]


In [23]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Criar a hora decimal (ex: 14:30 = 14.5) para que o eixo X represente o perfil diário corretamente
df_vis['Hora_Decimal'] = df_vis['Date_Time'].dt.hour + (df_vis['Date_Time'].dt.minute / 60.0)

df_protos = df_vis[df_vis['Date'].isin(proto_dates)]
df_crits = df_vis[df_vis['Date'].isin(crit_dates)]

# ----- Gráfico 1: Irradiação (GHI) -----
fig_ghi = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=('Protótipos (Dias Típicos) - Irradiação GHI', 
                                    'Críticas (Dias Anômalos) - Irradiação GHI'))

for d in proto_dates:
    day_data = df_protos[df_protos['Date'] == d]
    fig_ghi.add_trace(go.Scatter(x=day_data['Hora_Decimal'], y=day_data['ghi'], 
                             mode='lines', name=f"Typ: {d}", line=dict(width=2.5)), row=1, col=1)

for d in crit_dates:
    day_data = df_crits[df_crits['Date'] == d]
    fig_ghi.add_trace(go.Scatter(x=day_data['Hora_Decimal'], y=day_data['ghi'], 
                             mode='lines', name=f"Anm: {d}", line=dict(width=2.5, dash='dot')), row=2, col=1)

fig_ghi.update_layout(height=650, title_text="Inspeção MMD-Critic: Irradiação (GHI) ao longo do dia")
fig_ghi.update_xaxes(title_text="Hora do Dia (0-24h)", row=2, col=1)
fig_ghi.update_yaxes(title_text="GHI (Escalado)", row=1, col=1)
fig_ghi.update_yaxes(title_text="GHI (Escalado)", row=2, col=1)
fig_ghi.show()

# ----- Gráfico 2: Temperatura Ambiente -----
fig_temp = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=('Protótipos (Dias Típicos) - Temperatura', 
                                    'Críticas (Dias Anômalos) - Temperatura'))

for d in proto_dates:
    day_data = df_protos[df_protos['Date'] == d]
    fig_temp.add_trace(go.Scatter(x=day_data['Hora_Decimal'], y=day_data['temp_amb'], 
                             mode='lines', name=f"Typ: {d}", line=dict(width=2.5)), row=1, col=1)

for d in crit_dates:
    day_data = df_crits[df_crits['Date'] == d]
    fig_temp.add_trace(go.Scatter(x=day_data['Hora_Decimal'], y=day_data['temp_amb'], 
                             mode='lines', name=f"Anm: {d}", line=dict(width=2.5, dash='dot')), row=2, col=1)

fig_temp.update_layout(height=650, title_text="Inspeção MMD-Critic: Temperatura Ambiente ao longo do dia")
fig_temp.update_xaxes(title_text="Hora do Dia (0-24h)", row=2, col=1)
fig_temp.update_yaxes(title_text="Temp (Escalada)", row=1, col=1)
fig_temp.update_yaxes(title_text="Temp (Escalada)", row=2, col=1)
fig_temp.show()
